# 04 — SIL Streaming Simulation
# Giai đoạn 3 — Mục 3.3 — Mô phỏng xử lý luồng tín hiệu
**Mục đích**: minh họa pipeline xử lý liên tục từ tín hiệu đến chẩn đoán.

In [ ]:
from pathlib import Path
import numpy as np
import time
import pandas as pd
from common import io_utils, dsp, quantization, pipeline, config as cfg

In [ ]:
OUTPUT_DIR = Path("./outputs")
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

# Load model đã lượng tử hóa (chọn một fold làm ví dụ)

In [ ]:
model_path = MODELS_DIR / "mlp_test_load_0.tflite"  # Thay bằng model thực tế
if not model_path.exists():
    # Thử tìm bất kỳ model nào
    model_files = list(MODELS_DIR.glob("*.tflite"))
    if model_files:
        model_path = model_files[0]
        print(f"Sử dụng model: {model_path.name}")
    else:
        raise FileNotFoundError("Không tìm thấy file .tflite nào.")

with open(model_path, 'rb') as f:
    tflite_bytes = f.read()

# Load scaler

In [ ]:
scaler_path = MODELS_DIR / model_path.name.replace('.tflite', '_scaler.pkl')

# Đọc dữ liệu mẫu

In [ ]:
manifest = pd.read_csv(TABLES_DIR / "manifest_filtered.csv")
fp = pipeline.pick_file(manifest, label="OR", load_hp=0, diameter_mils=21)
signal = io_utils.load_de_signal(Path(fp))

In [ ]:
# Hàm xử lý một khung tín hiệu

BAND_HZ = (3200, 3800)
LP_CUTOFF = 500
WINDOW_SIZE = 2048
STRIDE = 1024